# mid file feature extraction

In [15]:
import os
import numpy as np
import pretty_midi
from tqdm import tqdm

In [16]:
DATA_DIR = "../Composer_Dataset/NN_midi_files_extended"
SETS = ['train', 'dev', 'test']
FIXED_SEQUENCE_LENGTH = 5000  # truncate or pad to this many notes

def extract_lstm_features(midi_path, max_length=FIXED_SEQUENCE_LENGTH):
    try:
        midi = pretty_midi.PrettyMIDI(midi_path)
        notes = []
        for inst in midi.instruments:
            if inst.is_drum:
                continue
            notes.extend(inst.notes)
        if not notes:
            return None  # skip empty or drum-only files

        notes.sort(key=lambda n: n.start)
        sequence = []
        prev_start = 0.0
        for note in notes:
            pitch = note.pitch
            velocity = note.velocity
            duration = note.end - note.start
            time_since_last = note.start - prev_start
            sequence.append([pitch, velocity, duration, time_since_last])
            prev_start = note.start

        if len(sequence) >= max_length:
            sequence = sequence[:max_length]
        else:
            reps = int(np.ceil(max_length / len(sequence)))
            sequence = (sequence * reps)[:max_length]

        return np.array(sequence, dtype=np.float32)

    except Exception as e:
        print(f"Failed to process {midi_path}: {e}")
        return None

# Data Containers
X = {'train': [], 'dev': [], 'test': []}
y = {'train': [], 'dev': [], 'test': []}
composer_to_idx = {}

# Dataset Processing
for split in SETS:
    split_path = os.path.join(DATA_DIR, split)
    print(f"Processing {split.upper()} set...")

    for composer_name in os.listdir(split_path):
        composer_path = os.path.join(split_path, composer_name)
        if not os.path.isdir(composer_path):
            continue

        # Assign label
        if composer_name not in composer_to_idx:
            composer_to_idx[composer_name] = len(composer_to_idx)
        label = composer_to_idx[composer_name]

        for fname in tqdm(os.listdir(composer_path), desc=f"{split}/{composer_name}"):
            if not fname.endswith(".mid"):
                continue
            midi_path = os.path.join(composer_path, fname)
            features = extract_lstm_features(midi_path)
            if features is not None:
                X[split].append(features)
                y[split].append(label)

# Convert to numpy arrays
for split in SETS:
    X[split] = np.stack(X[split])
    y[split] = np.array(y[split])

    print(f"{split}: {X[split].shape}, labels: {y[split].shape}")

print("Composer to index mapping:", composer_to_idx)

Processing TRAIN set...


train/bartok: 100%|██████████| 42/42 [00:01<00:00, 23.72it/s]


Processing DEV set...


dev/bartok: 100%|██████████| 5/5 [00:00<00:00, 68.27it/s]


Processing TEST set...


test/bartok: 100%|██████████| 5/5 [00:00<00:00, 45.16it/s]

train: (369, 5000, 4), labels: (369,)
dev: (35, 5000, 4), labels: (35,)
test: (35, 5000, 4), labels: (35,)
✅ Done.
Composer to index mapping: {'mozart': 0, 'chopin': 1, 'handel': 2, 'byrd': 3, 'schumann': 4, 'mendelssohn': 5, 'hummel': 6, 'bach': 7, 'bartok': 8}


In [ ]:
# Save as .npy
np.save("X_train_lstm.npy", X['train'])
np.save("y_train_lstm.npy", y['train'])
np.save("X_dev_lstm.npy", X['dev'])
np.save("y_dev_lstm.npy", y['dev'])
np.save("X_test_lstm.npy", X['test'])
np.save("y_test_lstm.npy", y['test'])